In [ ]:
!pip install category_encoders

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 5.7 MB/s eta 0:00:00


In [ ]:

pip install category_encoders

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 5.8 MB/s eta 0:00:00


In [ ]:
pip install joblib

In [7]:
import pandas as pd
import numpy as np
import xgboost as xgb
import joblib
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import category_encoders as ce
import warnings

warnings.filterwarnings('ignore')

def train_xgboost_colab_pro_nolog(file_path, output_excel_name,model_pkl_adi="xgboost_sampiyon_model.pkl", encoder_pkl_adi="target_encoder.pkl"):
    print("1. Veri Yükleniyor ve Hazırlanıyor...")
    df = pd.read_excel(file_path)

    # Sütun ismi uyuşmazlıklarını otomatik çöz
    df.rename(columns={'Toplam Desi': 'Toplam_Desi'}, inplace=True)
    df['Tarih'] = pd.to_datetime(df['Tarih'])

    # =====================================================================
    # 🚨 SİSTEM ZIRHI 1: SENTETİK VE EKSİK VERİ TEMİZLİĞİ 🚨
    if 'Sentetik_Mi' in df.columns:
        gercek_satir_sayisi = len(df)
        df = df[df['Sentetik_Mi'] == 0].copy()
        print(f"   -> {gercek_satir_sayisi - len(df)} adet Sentetik satır sistemden tamamen atıldı.")

    df = df.dropna(subset=['Toplam_Desi'])
    # =====================================================================

    # KUSURSUZ ZAMAN KESİĞİ (Son 7 Günü Teste Ayırma)
    max_date = df['Tarih'].max()
    split_date = max_date - pd.Timedelta(days=6)

    print(f"   Eğitim (Train) Bitiş: {split_date - pd.Timedelta(days=1):%Y-%m-%d}")
    print(f"   Test Seti Aralığı: {split_date:%Y-%m-%d} / {max_date:%Y-%m-%d}")

    # Veriyi Train and Test olarak böl
    train_df = df[df['Tarih'] < split_date].copy()
    test_df = df[df['Tarih'] >= split_date].copy()

    output_df = test_df.copy()

    # =====================================================================
    # 🚨 SİSTEM ZIRHI 2: ANALİZ SÜTUNLARINI VE METİNLERİ GİZLE 🚨
    drop_cols = ['Tarih', 'Toplam_Desi', 'Rota_Adi', 'Sentetik_Mi']
    cols_to_drop = [c for c in drop_cols if c in df.columns]

    eski_lag_cols = [c for c in df.columns if 'eski_lag' in c.lower()]
    if eski_lag_cols:
        print(f"   -> Denetim amaçlı açılan eski_lag sütunları modelden gizleniyor: {eski_lag_cols}")
        cols_to_drop.extend(eski_lag_cols)

    X_train = train_df.drop(columns=cols_to_drop)
    # 🔥 LOGARİTMA İPTAL EDİLDİ: Hedef artık saf desidir.
    y_train_gercek = train_df['Toplam_Desi']

    X_test = test_df.drop(columns=cols_to_drop)
    y_test_gercek = test_df['Toplam_Desi']
    # =====================================================================

    # =====================================================================
    # 🔥 TARGET ENCODING (LOGARİTMASIZ) 🔥
    # =====================================================================
    print("\n2. Target Encoding Uygulanıyor (Logaritmasız)...")

    kategorik_sutunlar = []
    if 'Rota_ID' in X_train.columns:
        kategorik_sutunlar.append('Rota_ID')
    if 'Çıkış Transfer Merkezi' in X_train.columns:
        kategorik_sutunlar.append('Çıkış Transfer Merkezi')
    if 'Varış Transfer Merkezi' in X_train.columns:
        kategorik_sutunlar.append('Varış Transfer Merkezi')

    if kategorik_sutunlar:
        target_encoder = ce.TargetEncoder(cols=kategorik_sutunlar, smoothing=10)

        # 1. KURAL: Artık logaritmik değil, devasa saf desi ortalamaları hesaplanacak.
        X_train = target_encoder.fit_transform(X_train, y_train_gercek)
        X_test = target_encoder.transform(X_test)
        print(f"   -> {kategorik_sutunlar} sütunları Target Encoding ile dönüştürüldü.")
    else:
        print("   -> Dönüştürülecek kategorik sütun (Rota_ID vb.) bulunamadı.")
    # =====================================================================

    print(f"\nEğitime Girecek Nihai Özellikler (Features): {X_train.columns.tolist()}")

    print("\n3. XGBoost GPU Hiperparametre Avı Başlıyor (Saf Desi Uzayında)...")

    param_grid = {
        'n_estimators': np.arange(100, 550, 50),
        'learning_rate': [0.01, 0.02, 0.03, 0.05, 0.08, 0.1, 0.15],
        'max_depth': [4, 5, 6, 7, 8, 9],
        'subsample': [0.65, 0.75, 0.85, 0.95, 1.0],
        'colsample_bytree': [0.65, 0.75, 0.85, 0.95, 1.0],
        'min_child_weight': [1, 3, 5, 7, 10],
        'gamma': [0, 1, 5, 10]
    }

    xgb_model = xgb.XGBRegressor(
        objective='reg:squarederror',
        random_state=42,
        tree_method='hist',
        device='cuda',
        n_jobs=-1
    )

    random_search = RandomizedSearchCV(
        estimator=xgb_model,
        param_distributions=param_grid,
        n_iter=150,
        scoring='neg_mean_absolute_error',
        cv=3,
        verbose=1,
        random_state=42,
        n_jobs=-1
    )

    try:
        random_search.fit(X_train, y_train_gercek)
    except Exception as e:
        print(f"\n GPU Hatası alındı, CPU'ya geçiliyor... Detay: {e}")
        xgb_model.set_params(device='cpu')
        random_search.fit(X_train, y_train_gercek)

    best_model = random_search.best_estimator_

    print("\n[ŞAMPİYON MODEL BULUNDU] En İyi Parametreler:")
    for param, value in random_search.best_params_.items():
        print(f"   * {param}: {value}")

    # TAHMİN BÖLÜMÜ (Ters Dönüşüm Yok)
    print("\n4. Test Seti Üzerinde Tahminler Yapılıyor...")

    # 🔥 DÖNÜŞÜM İPTAL: Tahminler zaten direkt saf desi olarak çıkıyor.
    y_pred_gercek = best_model.predict(X_test)
    y_pred_gercek = np.maximum(y_pred_gercek, 0)

    # 🔥 KRİTİK: MODEL VE ENCODER KAYIT BLOĞU 🔥
    # =====================================================================
    print(f"\n6. Model ve Encoder Diske Kaydediliyor...")

    # Encoder'ı Kaydet
    if 'target_encoder' in locals():
        joblib.dump(target_encoder, encoder_pkl_adi)
        print(f"   -> Target Encoder başarıyla kaydedildi: {encoder_pkl_adi}")

    # Best Model'i Kaydet
    joblib.dump(best_model, model_pkl_adi)
    print(f"   -> XGBoost Modeli başarıyla kaydedildi: {model_pkl_adi}")
    print(f"   -> Bu dosyaları ( {model_pkl_adi} ve {encoder_pkl_adi} ) bilgisayarına indirip VS Code projesine atabilirsin.")
    # =====================================================================

    # EXCEL ÇIKTISI
    output_df['Tahmin_Desi'] = np.round(y_pred_gercek, 2)

    # BAŞARI METRİKLERİ
    mae = mean_absolute_error(y_test_gercek, y_pred_gercek)
    rmse = np.sqrt(mean_squared_error(y_test_gercek, y_pred_gercek))
    r2 = r2_score(y_test_gercek, y_pred_gercek)
    wape = np.sum(np.abs(y_test_gercek - y_pred_gercek)) / (np.sum(y_test_gercek) + 1e-9)

    print("-" * 50)
    print("🏆 MODEL GENEL BAŞARI RAPORU (TEST SETİ)")
    print("-" * 50)
    print(f"MAE (Ortalama Mutlak Hata): {mae:.2f} Desi")
    print(f"RMSE (Kök Ortalama Kare Hata): {rmse:.2f} Desi")
    print(f"R2 Skoru (Açıklanan Varyans): {r2:.4f}")
    print(f"WAPE (Hata Oranı): %{wape * 100:.2f}")
    print(f"Doğruluk Oranı (1 - WAPE): %{(1 - wape) * 100:.2f}")
    print("-" * 50)

    # FEATURE IMPORTANCE
    print("\n5. Özelliklerin Gain (Kazanç) Analizi:")
    importance_dict = best_model.get_booster().get_score(importance_type='gain')
    sorted_importance = sorted(importance_dict.items(), key=lambda x: x[1], reverse=True)

    print(f"{'Özellik (Feature)':<30} | {'Gain Kazancı':<15}")
    print("-" * 50)
    for k, v in sorted_importance[:20]:
        print(f"{k:<30} | {v:,.2f}")

    # EXCEL ÇIKTISI
    output_df['Tahmin_Desi'] = np.round(y_pred_gercek, 2)
    output_df['Mutlak_Hata'] = np.abs(output_df['Toplam_Desi'] - output_df['Tahmin_Desi'])

    cols = ['Tarih', 'Rota_Adi', 'Toplam_Desi', 'Tahmin_Desi', 'Mutlak_Hata']
    mevcut_cols = [c for c in cols if c in output_df.columns]
    other_cols = [c for c in output_df.columns if c not in mevcut_cols]
    output_df = output_df[mevcut_cols + other_cols]

    output_df.to_excel(output_excel_name, index=False)
    print(f"\n[İŞLEM TAMAM] Test seti detayları '{output_excel_name}' dosyasına kaydedildi!")

# ==========================================
# COLAB ÇALIŞTIRMA BÖLÜMÜ
# ==========================================
GIRDI_MATRISI = "/content/MODEL1_READY_MASTER_DATASET_2_ESKİ.xlsx"
CIKTI_EXCEL = "/content/TEST_SONUCLARI_Gercek_vs_Tahmin_NOLOG_TARGET.xlsx"

train_xgboost_colab_pro_nolog(GIRDI_MATRISI, CIKTI_EXCEL)

1. Veri Yükleniyor ve Hazırlanıyor...
   -> 590 adet Sentetik satır sistemden tamamen atıldı.
   Eğitim (Train) Bitiş: 2026-05-03
   Test Seti Aralığı: 2026-05-04 / 2026-05-10
   -> Denetim amaçlı açılan eski_lag sütunları modelden gizleniyor: ['eski_lag_7_tatil', 'eski_lag_7_kriz', 'eski_lag_7_tatil_sonrasi', 'eski_lag_14_tatil', 'eski_lag_14_kriz', 'eski_lag_14_tatil_sonrasi']

2. Target Encoding Uygulanıyor (Logaritmasız)...
   -> ['Rota_ID'] sütunları Target Encoding ile dönüştürüldü.

Eğitime Girecek Nihai Özellikler (Features): ['Rota_ID', 'lag_7', 'lag_14', 'is_holiday', 'tatil_sonrasi_mi', 'Kriz_Mi', 'Ay', 'Ayin_Gunu', 'Haftanin_Gunu', 'rolling_7_mean', 'rolling_7_std', 'rolling_14_mean', 'rolling_14_std']

3. XGBoost GPU Hiperparametre Avı Başlıyor (Saf Desi Uzayında)...
Fitting 3 folds for each of 150 candidates, totalling 450 fits

[ŞAMPİYON MODEL BULUNDU] En İyi Parametreler:
   * subsample: 0.65
   * n_estimators: 400
   * min_child_weight: 7
   * max_depth: 5
   * learnin